## Per-article download via bioRxiv API

For testing or fetching a handful of articles by DOI. Downloads JATS XML and rendered JPEG figures (not the tiff original from .meca). Slower download and lower resolution images, but works without AWS credentials.

In [ ]:
 # Here are list of ids to reproduce current work. This will allow to run demo and run the pipeline on golden set

 # Worked example from the BFG demo
DEMO_EXAMPLE = [
    "10.1101/2024.01.24.577109",
]

# ids for golden set. Set of CC0 and CCBY articles.
GOLD_SET_DOI = [
    "10.1101/2024.01.24.577109",
]

In [13]:
import time
import xml.etree.ElementTree as ET
from pathlib import Path
from urllib.parse import urlparse

import httpx

# Target DOIs  edit this list
DOIS = [
    "10.1101/2024.01.24.577109",   # worked example from the BFG demo
]
OUT_DIR = Path("./from_api")

SERVER = "biorxiv" # "biorxiv" or "medrxiv"
IMAGE_VARIANT = "large" # "large" | "medium" | "small"

REQUEST_DELAY_S = 1.0
TIMEOUT_S = 60.0
USER_AGENT = "cm3070-bfg/0.1 (biomedical figure research)"

_client = httpx.Client(
    headers={"User-Agent": USER_AGENT, "Accept": "*/*"},
    verify=False,
    timeout=TIMEOUT_S,
    follow_redirects=True,
)

def polite_get(url: str) -> httpx.Response:
    time.sleep(REQUEST_DELAY_S)
    r = _client.get(url)
    r.raise_for_status()
    return r

print(f"Output:    {OUT_DIR.resolve()}")

CA bundle: /Users/kkrt923/zscaler.pem
Output:    /Users/kkrt923/Projects/cm3070-final/data/from_api


In [14]:
API_BASE = "https://api.biorxiv.org"
_NS_XLINK = "http://www.w3.org/1999/xlink"

def fetch_by_doi(server: str, doi: str) -> list[dict]:
    url = f"{API_BASE}/details/{server}/{doi}"
    return polite_get(url).json().get("collection", [])

def extract_figures(xml_bytes: bytes) -> list[dict]:
    root = ET.fromstring(xml_bytes)
    figs: list[dict] = []
    for fig in root.iter("fig"):
        g = fig.find(".//graphic")
        href = g.get(f"{{{_NS_XLINK}}}href") if g is not None else None
        if not href:
            continue
        caption_el = fig.find(".//caption")
        caption = "".join(caption_el.itertext()).strip() if caption_el is not None else ""
        figs.append({"id": fig.get("id") or f"fig{len(figs)+1}",
                     "index": len(figs) + 1, "tif_href": href, "caption": caption})
    return figs

def _jats_dir(jats_url: str) -> str:
    p = urlparse(jats_url)
    fname = p.path.rsplit("/", 1)[-1]
    stem = fname.removesuffix(".source.xml")
    parent = p.path.rsplit("/", 1)[0].replace("/content/", "/content/biorxiv/", 1)
    return f"{p.scheme}://{p.netloc}{parent}/{stem}"

def _slug(doi: str) -> str:
    return doi.replace("/", "_")

def download_article(meta: dict, out_root: Path, variant: str = "large") -> Path:
    doi = meta["doi"]
    version = meta.get("version", "1")
    jats_url = meta["jatsxml"]
    slug = f"{_slug(doi)}_v{version}"
    dest = out_root / slug
    dest.mkdir(parents=True, exist_ok=True)
    print(f"[{slug}] XML <- {jats_url}")
    xml_bytes = polite_get(jats_url).content
    (dest / "article.xml").write_bytes(xml_bytes)
    fig_dir = _jats_dir(jats_url)
    figs = extract_figures(xml_bytes)
    print(f"[{slug}] {len(figs)} figure(s)")
    for fig in figs:
        img_url = f"{fig_dir}/F{fig['index']}.{variant}.jpg"
        img_path = dest / f"F{fig['index']}.{variant}.jpg"
        try:
            data = polite_get(img_url).content
        except httpx.HTTPStatusError as e:
            print(f"  ! F{fig['index']}: HTTP {e.response.status_code}")
            continue
        img_path.write_bytes(data)
        print(f"  F{fig['index']:<2} {len(data):>8,} B")
    return dest

In [15]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

records = []
for doi in DOIS:
    versions = fetch_by_doi(SERVER, doi)
    if not versions:
        print(f"no records for {doi}")
        continue
    records.append(max(versions, key=lambda r: int(r.get("version", 1))))

print(f"{len(records)} records to download\n")
for r in records:
    download_article(r, OUT_DIR, variant=IMAGE_VARIANT)

print(f"\nSaved : {OUT_DIR.resolve()}")

1 record(s) to download

[10.1101_2024.01.24.577109_v4] XML <- https://www.biorxiv.org/content/early/2026/05/28/2024.01.24.577109.source.xml
[10.1101_2024.01.24.577109_v4] 8 figure(s)
  F1    57,585 B
  F2   133,171 B
  F3   122,315 B
  F4    83,782 B
  F5   144,568 B
  F6   126,115 B
  F7   107,154 B
  F8    88,211 B

Done. Saved under: /Users/kkrt923/Projects/cm3070-final/data/from_api
